# Ingest — Copilot Studio

Reads the three Power Platform admin center exports from `Files/landing/studio/` into
`studio_tenant_daily`, `studio_agent` and `studio_user`.

## The awkward bit: two of these have no date

`StudioTenantDaily.csv` carries `Usage Date`, so it merges cleanly on the day.

`StudioPerAgent.csv` and `StudioPerUser.csv` do **not**. They are period aggregates for whatever
window the portal had selected — PPAC shows month-to-date plus the last two full months. Appending
them blindly would double-count on every re-run; overwriting them would throw away last month.

So this stamps each load with a `snapshot_month` and merges on that. Re-run in the same month and
the row updates; run next month and a new row appears. **The result is per-agent history from a
source that has none** — which is most of the point of running this in Fabric at all.

One consequence worth understanding: the snapshot is a *cumulative month-to-date* figure, not a
monthly increment. Comparing two snapshots within the same month shows growth to date, not new
consumption. CreditLens treats these as period totals for exactly this reason and never plots them
on a time axis.

In [ ]:
LANDING = "Files/landing/studio"

# Which month these agent/user aggregates belong to. Defaults to the month the
# notebook runs; override when backfilling an export you took earlier.
SNAPSHOT_MONTH = None      # e.g. "2026-07-01"

TBL_TENANT = "studio_tenant_daily"
TBL_AGENT = "studio_agent"
TBL_USER = "studio_user"

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import date
import re


def norm(name):
    return re.sub(r"[ _\-/]", "", name).lower()


def pick(df, *aliases):
    """Actual column name matching any alias, ignoring case and separators.

    The PPAC headers contain spaces and a slash ('AI Feature/Billable Feature'),
    and are the sort of thing that gets tidied up between releases.
    """
    lookup = {norm(c): c for c in df.columns}
    for a in aliases:
        if norm(a) in lookup:
            return lookup[norm(a)]
    return None


def read(pattern):
    try:
        df = (spark.read.option("header", True).option("inferSchema", False)
              .csv(f"{LANDING}/{pattern}"))
        return df if df.count() else None
    except Exception as e:
        print(f"  {pattern}: not found ({type(e).__name__})")
        return None


def merge(df, table, keys):
    if spark.catalog.tableExists(table):
        before = spark.table(table).count()
        cond = " AND ".join(f"t.{k} <=> s.{k}" for k in keys)
        (DeltaTable.forName(spark, table).alias("t")
            .merge(df.alias("s"), cond)
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
        after = spark.table(table).count()
        print(f"  {table}: {before:,} -> {after:,}  (+{after - before:,})")
    else:
        df.write.format("delta").saveAsTable(table)
        print(f"  {table}: created with {df.count():,} rows")


snapshot = (date.fromisoformat(SNAPSHOT_MONTH) if SNAPSHOT_MONTH
            else date.today().replace(day=1))
print(f"snapshot month: {snapshot}")

## Tenant daily

The only one of the three with a real date, and the only one CreditLens plots over time.

In [ ]:
raw = read("*Tenant*.csv")
if raw is None:
    print("no tenant export - skipping")
else:
    tenant = raw.select(
        F.col(pick(raw, "BillingPlan Id")).cast("string").alias("billing_plan_id"),
        F.col(pick(raw, "BillingPlan Name")).cast("string").alias("billing_plan_name"),
        F.col(pick(raw, "Environment Id")).cast("string").alias("environment_id"),
        F.col(pick(raw, "Environment Name")).cast("string").alias("environment_name"),
        F.col(pick(raw, "Capacity Type")).cast("string").alias("capacity_type"),
        F.col(pick(raw, "Entitled Quantity")).cast("double").alias("entitled_quantity"),
        F.col(pick(raw, "Prepaid Consumed Quantity")).cast("double").alias("prepaid_consumed"),
        F.col(pick(raw, "Pay as you go Consumed Quantity")).cast("double").alias("payg_consumed"),
        # PPAC writes US-style M/d/yyyy with a time component
        F.to_date(F.col(pick(raw, "Usage Date")), "M/d/yyyy H:mm").alias("usage_date"),
    ).withColumn("_loaded_at", F.current_timestamp())

    bad = tenant.filter(F.col("usage_date").isNull()).count()
    if bad:
        print(f"  ! {bad:,} rows have an unparseable Usage Date - check the format")

    print(f"  {tenant.count():,} rows, "
          f"{tenant.select('environment_id').distinct().count()} environments")
    merge(tenant, TBL_TENANT,
          ["usage_date", "environment_id", "billing_plan_id", "capacity_type"])

## Per agent

No date in the file, so the snapshot month becomes part of the key.

In [ ]:
raw = read("*Agent*.csv")
if raw is None:
    print("no per-agent export - skipping")
else:
    agent = raw.select(
        F.lit(snapshot).cast("date").alias("snapshot_month"),
        F.col(pick(raw, "Agent Name")).cast("string").alias("agent_name"),
        F.col(pick(raw, "Agent Id")).cast("string").alias("agent_id"),
        F.col(pick(raw, "Product")).cast("string").alias("product"),
        F.col(pick(raw, "AI Feature/Billable Feature", "Billable Feature"))
            .cast("string").alias("billable_feature"),
        F.col(pick(raw, "Billed credit")).cast("double").alias("billed_credit"),
        F.col(pick(raw, "Non-billed credit")).cast("double").alias("non_billed_credit"),
        F.col(pick(raw, "Channel")).cast("string").alias("channel"),
        F.col(pick(raw, "Knowledge Sources")).cast("string").alias("knowledge_sources"),
        F.col(pick(raw, "Tool Used")).cast("string").alias("tool_used"),
        F.col(pick(raw, "LLM Model")).cast("string").alias("llm_model"),
        F.col(pick(raw, "Scenario Name")).cast("string").alias("scenario_name"),
        F.col(pick(raw, "Environment Id")).cast("string").alias("environment_id"),
        F.col(pick(raw, "Environment Name")).cast("string").alias("environment_name"),
    ).withColumn("_loaded_at", F.current_timestamp())

    print(f"  {agent.count():,} rows, "
          f"{agent.select('agent_id').distinct().count()} agents")
    # An agent appears once per feature per channel per environment, so all four
    # belong in the key - dropping any of them silently loses rows on merge.
    merge(agent, TBL_AGENT,
          ["snapshot_month", "agent_id", "billable_feature", "channel", "environment_id"])

## Per user

In [ ]:
raw = read("*User*.csv")
if raw is None:
    print("no per-user export - skipping")
else:
    user = raw.select(
        F.lit(snapshot).cast("date").alias("snapshot_month"),
        F.col(pick(raw, "User Id")).cast("string").alias("user_id"),
        F.lower(F.trim(F.col(pick(raw, "User Email")))).alias("user_email"),
        F.col(pick(raw, "Agent Id")).cast("string").alias("agent_id"),
        F.col(pick(raw, "Agent Name")).cast("string").alias("agent_name"),
        F.col(pick(raw, "Billable credit used")).cast("double").alias("billable_credit_used"),
        F.col(pick(raw, "Credits used")).cast("double").alias("credits_used"),
        F.upper(F.trim(F.col(pick(raw, "M365 Copilot Licensed"))))
            .isin("TRUE", "YES", "1").alias("m365_copilot_licensed"),
    ).withColumn("_loaded_at", F.current_timestamp())

    print(f"  {user.count():,} rows, "
          f"{user.select('user_email').distinct().count()} users")
    merge(user, TBL_USER, ["snapshot_month", "user_id", "agent_id"])

## Check

The tenant file is the source of truth for totals. The per-agent and per-user files are
attributions of it, and they will not add up to it — agent and environment consumption carries no
user, so the per-user total is always the smaller number. That gap is expected and CreditLens says
so on the page rather than hiding it.

In [ ]:
for t in (TBL_TENANT, TBL_AGENT, TBL_USER):
    if spark.catalog.tableExists(t):
        n = spark.table(t).count()
        print(f"{t:24s} {n:>8,} rows")
    else:
        print(f"{t:24s}        - not created")

if spark.catalog.tableExists(TBL_TENANT):
    spark.sql(f"""
        SELECT  MIN(usage_date) AS earliest, MAX(usage_date) AS latest,
                COUNT(DISTINCT usage_date) AS days,
                ROUND(SUM(prepaid_consumed)) AS prepaid,
                ROUND(SUM(payg_consumed))    AS payg
        FROM    {TBL_TENANT}
    """).show(truncate=False)